# Lab 3 — Naive Bayes

Notebook hoàn thành các phần thực hành trong `Lab03_Naive_Bayes.pdf`: định lý Bayes, túi từ, lọc tin nhắn rác với `MultinomialNB`, làm trơn Laplace, phân loại số liệu liên tục với `GaussianNB`, so sánh với hồi quy logistic và **đầy đủ 6 bài tập cuối tài liệu**.

Đặt notebook cùng thư mục với `tin_nhan.csv` và `sinh_vien.csv`. Nhãn `la_rac=1` là tin rác; `la_rac=0` là tin thường.

In [ ]:
from pathlib import Path
from collections import Counter, defaultdict
import math

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, confusion_matrix, precision_score,
                             recall_score)
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB, MultinomialNB

DATA_DIR = Path.cwd()
for filename in ("tin_nhan.csv", "sinh_vien.csv"):
    if not (DATA_DIR / filename).exists():
        raise FileNotFoundError(f"Không tìm thấy {filename}; hãy chạy notebook từ thư mục chứa hai CSV.")

df_tin = pd.read_csv(DATA_DIR / "tin_nhan.csv")
df_sv = pd.read_csv(DATA_DIR / "sinh_vien.csv")
print("tin_nhan:", df_tin.shape, "| sinh_vien:", df_sv.shape)
display(df_tin.head(), df_sv.head())

## 1. Định lý Bayes bằng công thức và cách đếm

In [ ]:
def bayes(tien_nghiem, kha_nang, kha_nang_nguoc):
    """P(A|B) = P(B|A)P(A) / P(B), tính mẫu số bằng xác suất toàn phần."""
    tu_so = kha_nang * tien_nghiem
    mau_so = tu_so + kha_nang_nguoc * (1 - tien_nghiem)
    return tu_so / mau_so

# Bệnh hiếm: độ nhạy 99%, độ đặc hiệu 99%
p_benh = bayes(0.01, 0.99, 0.01)
print(f"P(có bệnh | dương tính) = {p_benh:.4f}")
print("Tỷ lệ mắc bệnh | Hậu nghiệm khi dương tính")
for prevalence in [0.001, 0.01, 0.05, 0.10, 0.50]:
    print(f"{prevalence:>8.3f} | {bayes(prevalence, 0.99, 0.01):.4f}")

# Sản phẩm lỗi từ dây chuyền B
p_day_b = bayes(0.20, 0.05, 0.01)
print(f"\nP(dây B | sản phẩm lỗi) = {p_day_b:.4f}")
print("Kiểm tra bằng đếm: A có 8 lỗi, B có 10 lỗi; tỷ lệ của B = 10/18 =", round(10/18, 4))

**Diễn giải:** dù xét nghiệm có độ nhạy và độ đặc hiệu 99%, hậu nghiệm chỉ là 0.5 khi bệnh phổ biến 1% vì số người không bệnh rất lớn. Ví dụ hai dây chuyền cho xác suất sản phẩm lỗi đến từ B là 0.5556.

## 2. Khám phá dữ liệu và biến tin nhắn thành túi từ

In [ ]:
print("Kích thước:", df_tin.shape)
print("Số tin theo nhãn (0 thường, 1 rác):\n", df_tin["la_rac"].value_counts().sort_index())
print(f"Tỷ lệ tin rác: {df_tin['la_rac'].mean():.4f}")
print("\nBa tin đầu:")
for _, row in df_tin.head(3).iterrows():
    print(f"[{row.la_rac}] {row.noi_dung}")

# Tách theo khoảng trắng để giữ từ một ký tự và cách token hóa của tài liệu.
vectorizer = CountVectorizer(token_pattern=r"(?u)\S+")
X_bow = vectorizer.fit_transform(df_tin["noi_dung"])
y_tin = df_tin["la_rac"]
print("\nMa trận túi từ:", X_bow.shape, "| số từ:", len(vectorizer.vocabulary_))
counts = pd.DataFrame(X_bow.toarray(), columns=vectorizer.get_feature_names_out())
counts["la_rac"] = y_tin.to_numpy()
word_totals = counts.groupby("la_rac").sum().T
word_totals.columns = ["thuong", "rac"]
print("\n10 từ nhiều nhất trong tin rác:")
display(word_totals.sort_values("rac", ascending=False).head(10))
print("10 từ nhiều nhất trong tin thường:")
display(word_totals.sort_values("thuong", ascending=False).head(10))

**Diễn giải:** mỗi hàng là một tin nhắn, mỗi cột là một từ, giá trị là số lần từ xuất hiện. Túi từ không giữ thứ tự từ; ở bài lọc rác, từ vựng vẫn mang tín hiệu hữu ích. `token_pattern=r"(?u)\S+"` giữ cả token chỉ có một ký tự như `ạ`. 

## 3. Huấn luyện và đánh giá bộ lọc tin rác

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_bow, y_tin, test_size=0.25, random_state=17, stratify=y_tin
)
spam_model = MultinomialNB(alpha=1.0).fit(X_train, y_train)
y_pred = spam_model.predict(X_test)
tn, fp, fn, tp = confusion_matrix(y_test, y_pred, labels=[0, 1]).ravel()
print(f"Số mẫu train/test: {len(y_train)}/{len(y_test)}")
print(f"TN={tn}, FP={fp}, FN={fn}, TP={tp}")
print(f"Accuracy={accuracy_score(y_test, y_pred):.4f}")
print(f"Precision={precision_score(y_test, y_pred, zero_division=0):.4f}")
print(f"Recall={recall_score(y_test, y_pred, zero_division=0):.4f}")

sample_messages = [
    "trung thuong khuyen mai nhanh tay click ngay",
    "hop nhom bao cao chieu mai phong 305",
    "ban gui giup em tai lieu bai tap nhe",
]
for message, probability in zip(sample_messages, spam_model.predict_proba(vectorizer.transform(sample_messages))[:, 1]):
    print(f"{'RAC' if probability >= 0.5 else 'THUONG':7s} P(rác)={probability:.4f} | {message}")

**Diễn giải:** precision đo tỷ lệ dự đoán rác thực sự là rác; recall đo tỷ lệ tin rác được bắt. Với bộ lọc thư, cần xem cả hai vì nhầm tin thường thành rác có thể làm mất thư quan trọng. Xác suất Naive Bayes có thể quá tự tin do giả định các từ độc lập.

## 4. Vấn đề số không và tự cài đặt Laplace

In [ ]:
toy_messages = [
    ("trung thuong ngay", 1),
    ("khuyen mai giam gia", 1),
    ("trung thuong click", 1),
    ("hop nhom chieu mai", 0),
    ("bai tap nhom nop mai", 0),
    ("thay gui tai lieu", 0),
]
toy_vocab = sorted({word for message, _ in toy_messages for word in message.split()})
V = len(toy_vocab)
word_counts = {0: Counter(), 1: Counter()}
token_totals = {0: 0, 1: 0}
message_counts = Counter(label for _, label in toy_messages)
for message, label in toy_messages:
    tokens = message.split()
    word_counts[label].update(tokens)
    token_totals[label] += len(tokens)

print(f"Từ điển ({V} từ):", ", ".join(toy_vocab))
print("Từ | rác | thường")
for word in toy_vocab:
    print(f"{word:10s} | {word_counts[1][word]:3d} | {word_counts[0][word]:3d}")

def manual_nb_scores(message, alpha):
    scores = {}
    n_messages = sum(message_counts.values())
    for label in (0, 1):
        # Log prior + tổng log likelihood; tránh tích xác suất gây underflow.
        score = math.log(message_counts[label] / n_messages)
        for word in message.split():
            if word in word_counts[label] or word in toy_vocab:
                probability = (word_counts[label][word] + alpha) / (token_totals[label] + alpha * V)
                if probability == 0:
                    score = float("-inf")
                    break
                score += math.log(probability)
        scores[label] = score
    return scores

query = "trung thuong bai tap"
for alpha in (0.0, 1.0):
    scores = manual_nb_scores(query, alpha)
    print(f"\nalpha={alpha}")
    for label, score in scores.items():
        print(f"Lớp {label}: log-score={score:.6f}")
    print("Dự đoán:", max(scores, key=scores.get), "(0 thường, 1 rác)")

Khi `alpha=0`, từ chưa từng gặp ở một lớp cho log-score `-inf`, khiến mô hình dễ bị một từ khóa cả lớp. Laplace dùng `(đếm + α)/(tổng từ lớp + αV)` để mọi từ trong từ điển có xác suất dương. Cộng log-probability cũng tránh tích nhiều số nhỏ bị tràn dưới.

## 5. GaussianNB so với hồi quy logistic

In [ ]:
X_sv = df_sv[["gio_on", "diem_giua_ky"]]
y_sv = df_sv["qua_mon"]
Xsv_train, Xsv_test, ysv_train, ysv_test = train_test_split(
    X_sv, y_sv, test_size=0.25, random_state=17, stratify=y_sv
)
models = [("GaussianNB", GaussianNB()), ("Hồi quy logistic", LogisticRegression())]
for name, model in models:
    model.fit(Xsv_train, ysv_train)
    print(f"{name}: accuracy={accuracy_score(ysv_test, model.predict(Xsv_test)):.4f}")

gnb = GaussianNB().fit(Xsv_train, ysv_train)
print("\nThống kê GaussianNB đã học (theo lớp [0, 1]):")
for idx, label in enumerate(gnb.classes_):
    print(f"Lớp {label}: trung bình={gnb.theta_[idx].round(2)}, độ lệch chuẩn={np.sqrt(gnb.var_[idx]).round(2)}")
print("Prior lớp [0, 1]:", gnb.class_prior_.round(4))
new_students = pd.DataFrame({"gio_on": [5, 13, 20, 28], "diem_giua_ky": [4.0, 6.0, 7.0, 9.0]})
new_students["P(qua_mon)"] = gnb.predict_proba(new_students)[:, list(gnb.classes_).index(1)]
display(new_students)

**Diễn giải:** GaussianNB giả định đặc trưng liên tục trong mỗi lớp tuân theo phân phối chuẩn. Logistic regression học ranh giới phân lớp; GaussianNB còn ước lượng trung bình và phương sai theo lớp. Hai mô hình được so sánh trên cùng một phép chia train/test.

## 6. Bài tập 1 — Bayes với ba nguyên nhân

In [ ]:
groups = ["học đều", "học dồn", "ít học"]
priors = np.array([0.50, 0.30, 0.20])
pass_likelihoods = np.array([0.95, 0.70, 0.30])
unnormalized = priors * pass_likelihoods
posterior = unnormalized / unnormalized.sum()
result_ex1 = pd.DataFrame({"nhóm": groups, "P(nhóm)": priors,
                           "P(qua|nhóm)": pass_likelihoods,
                           "P(nhóm|đã qua)": posterior})
display(result_ex1)
print("Tổng hậu nghiệm:", posterior.sum())

**Kết quả:** áp dụng Bayes cho nhiều nhánh, `P(nhóm | đã qua) ∝ P(qua | nhóm)P(nhóm)`, rồi chuẩn hóa để tổng bằng 1.

## 7. Bài tập 2 — Từ tố cáo mạnh nhất

In [ ]:
full_text_vectorizer = CountVectorizer(token_pattern=r"(?u)\S+")
X_all_text = full_text_vectorizer.fit_transform(df_tin["noi_dung"])
full_nb = MultinomialNB(alpha=1.0).fit(X_all_text, y_tin)
feature_names = full_text_vectorizer.get_feature_names_out()
# Chênh lệch log P(từ|rác) - log P(từ|thường)
log_odds = full_nb.feature_log_prob_[1] - full_nb.feature_log_prob_[0]
word_evidence = pd.DataFrame({"từ": feature_names, "chênh_log_xác_suất": log_odds})
print("5 từ nghiêng mạnh về rác:")
display(word_evidence.nlargest(5, "chênh_log_xác_suất"))
print("5 từ nghiêng mạnh về thường:")
display(word_evidence.nsmallest(5, "chênh_log_xác_suất"))

## 8. Bài tập 3 — Ảnh hưởng của alpha

In [ ]:
alpha_rows = []
for alpha in [0.001, 0.1, 1.0, 10.0, 100.0]:
    model = MultinomialNB(alpha=alpha).fit(X_train, y_train)
    pred = model.predict(X_test)
    alpha_rows.append({"alpha": alpha,
                       "accuracy": accuracy_score(y_test, pred),
                       "precision": precision_score(y_test, pred, zero_division=0),
                       "recall": recall_score(y_test, pred, zero_division=0)})
display(pd.DataFrame(alpha_rows).set_index("alpha").round(4))

**Nhận xét:** alpha quá lớn làm xác suất từ giữa hai lớp xích lại gần nhau, nên khác biệt từ vựng ít tác động đến dự đoán; mô hình mất tín hiệu phân biệt và hiệu năng có thể giảm.

## 9. Bài tập 4 — Bộ lọc tự viết và đối chiếu thư viện

In [ ]:
# Chia trên văn bản gốc rồi fit từ điển chỉ trên tập huấn luyện.
text_train, text_test, y_custom_train, y_custom_test = train_test_split(
    df_tin["noi_dung"], y_tin, test_size=0.25, random_state=17, stratify=y_tin
)
train_tokens = [text.split() for text in text_train]
test_tokens = [text.split() for text in text_test]
vocab = sorted({token for row in train_tokens for token in row})
V_train = len(vocab)
class_docs = Counter(int(label) for label in y_custom_train)
class_token_counts = {0: Counter(), 1: Counter()}
class_token_totals = Counter()
for tokens, label in zip(train_tokens, y_custom_train):
    label = int(label)
    class_token_counts[label].update(tokens)
    class_token_totals[label] += len(tokens)

# Fit thư viện trên đúng cùng training text/vocabulary.
custom_vectorizer = CountVectorizer(token_pattern=r"(?u)\S+")
X_custom_train = custom_vectorizer.fit_transform(text_train)
X_custom_test = custom_vectorizer.transform(text_test)
library_model = MultinomialNB(alpha=1.0).fit(X_custom_train, y_custom_train)

classes = sorted(class_docs)
def predict_manual(tokens):
    scores = {}
    n_docs = len(y_custom_train)
    for label in classes:
        score = math.log(class_docs[label] / n_docs)
        denom = class_token_totals[label] + V_train  # alpha=1
        for token in tokens:
            if token in vocab:
                score += math.log((class_token_counts[label][token] + 1) / denom)
        scores[label] = score
    return max(scores, key=scores.get)

manual_predictions = np.array([predict_manual(tokens) for tokens in test_tokens])
library_predictions = library_model.predict(X_custom_test)
print("Manual accuracy:", round(accuracy_score(y_custom_test, manual_predictions), 4))
print("MultinomialNB accuracy:", round(accuracy_score(y_custom_test, library_predictions), 4))
print("Dự đoán trùng nhau:", np.mean(manual_predictions == library_predictions))

Bộ tự cài đặt dùng prior theo số văn bản mỗi lớp, đếm token, Laplace `α=1`, và tổng log xác suất. Vì cùng quy tắc với `MultinomialNB`, dự đoán và accuracy cần khớp hoặc rất gần nhau. Token ngoài từ điển train được bỏ qua, giống vectorizer.

## 10. Bài tập 5 — Dùng sai MultinomialNB cho dữ liệu sinh viên

In [ ]:
Xsv_train2, Xsv_test2, ysv_train2, ysv_test2 = train_test_split(
    X_sv, y_sv, test_size=0.25, random_state=17, stratify=y_sv
)
wrong_variant = MultinomialNB().fit(Xsv_train2, ysv_train2)
wrong_pred = wrong_variant.predict(Xsv_test2)
print(f"MultinomialNB accuracy={accuracy_score(ysv_test2, wrong_pred):.4f}")
print(f"GaussianNB accuracy={accuracy_score(ysv_test2, GaussianNB().fit(Xsv_train2, ysv_train2).predict(Xsv_test2)):.4f}")

**Giải thích:** chương trình chạy vì các giá trị giờ ôn và điểm thi đều không âm. Tuy nhiên MultinomialNB mô hình hóa số đếm/tần suất rời rạc; các đại lượng liên tục này không phải dữ liệu đếm. GaussianNB là lựa chọn phù hợp hơn về giả định thống kê, dù accuracy trên một lần chia có thể tình cờ tương đương.

## 11. Bài tập 6 — Lớp rác hiếm và prior

In [ ]:
ham = df_tin[df_tin["la_rac"] == 0]
spam_first_20 = df_tin[df_tin["la_rac"] == 1].head(20)
balanced_source = pd.concat([ham, spam_first_20], ignore_index=True)
X_imb_train_text, X_imb_test_text, y_imb_train, y_imb_test = train_test_split(
    balanced_source["noi_dung"], balanced_source["la_rac"],
    test_size=0.25, random_state=17, stratify=balanced_source["la_rac"]
)
imb_vectorizer = CountVectorizer(token_pattern=r"(?u)\S+")
X_imb_train = imb_vectorizer.fit_transform(X_imb_train_text)
X_imb_test = imb_vectorizer.transform(X_imb_test_text)
imb_model = MultinomialNB(alpha=1.0).fit(X_imb_train, y_imb_train)
imb_pred = imb_model.predict(X_imb_test)
print("Tỷ lệ nhãn train:", y_imb_train.value_counts(normalize=True).sort_index().round(3).to_dict())
print(f"Accuracy={accuracy_score(y_imb_test, imb_pred):.4f}")
print(f"Recall lớp rác={recall_score(y_imb_test, imb_pred, pos_label=1, zero_division=0):.4f}")
print("Ma trận nhầm lẫn [thường, rác]:\n", confusion_matrix(y_imb_test, imb_pred, labels=[0, 1]))

**Nhận xét:** accuracy một mình có thể tạo cảm giác mô hình tốt vì đa số mẫu là tin thường; mô hình đoán thường nhiều vẫn có thể bỏ lọt phần lớn tin rác. Recall lớp rác cho biết trực tiếp tỷ lệ tin rác được phát hiện. Cần xem thêm ma trận nhầm lẫn và metric theo từng lớp.

## Kết luận

- Bayes đổi xác suất tiên nghiệm và khả năng thành hậu nghiệm; tỷ lệ lớp nền có ảnh hưởng lớn.
- Túi từ + MultinomialNB phù hợp với đếm từ; Laplace ngăn xác suất bằng 0.
- GaussianNB phù hợp với đặc trưng liên tục theo giả định chuẩn.
- Đánh giá bộ lọc cần xét precision/recall cùng accuracy, đặc biệt khi lớp mất cân bằng.

Notebook sử dụng hai CSV trong cùng thư mục và có thể chạy từ đầu đến cuối bằng **Run All**.